# Moebius — Sphere-Congruence Quad Mesh Pipeline

Full pipeline in six stages:
0. **Setup** — imports, env check
1. **Load surface** — B-spline JSON, u/v range, flip options
2. **Initialisation** — sphere radii r(u,v), line congruence l(u,v)
3. **LC optimisation** — minimise l·cu, l·cv; enforce angle with n
4. **Torsal optimisation** — add torsal-plane constraints
5. **Remeshing** — export cross-field, run remesher, load result
6. **Post-optimisation** — sphere fit, support planarity, face regularity
7. **Save results** — OBJ files, sphere panels, state pickle

In [1]:
import sys, os
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

CHAKANA = (ROOT / "Chakana_Geo").resolve()
for sub in ["hanan/src", "kayviz/src"]:
    p = str(CHAKANA / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import matplotlib.pyplot as plt
import kayviz as kv

from moebius.pipeline import (
    MoebiusState, load_surface, sample_selection, compute_init,
    setup_bspline_optimizer, run_bspline_optimizer,
    setup_lc_optimizer, run_lc_optimizer,
    setup_torsal_optimizer, run_torsal_optimizer,
    save_remesh_input, run_remesher, load_remeshed_surface,
    setup_postopt_optimizer, run_postopt_optimizer,
    save_results, load_state,
)
from moebius.utils.bsplines import sample_bspline_surface, bspline_curvatures
from moebius.glyphs import spherical_panels_from_mesh, sphere_centers_radii_from_params
from scipy.interpolate import bisplev

kv.init()

[kayviz] Viewer running at http://127.0.0.1:8001


---
## 1 — Load B-spline surface

In [2]:
# ── parameters (edit these) ───────────────────────────────────────────────────
SURFACE_NAME    = "CMC"          # JSON filename without extension
EXPERIMENT_NAME = "experiment0"

# u/v sampling
U_SAMPLE = 25
V_SAMPLE = 25
INTERVAL_U = [0.0, 1.0]
INTERVAL_V = [0.0, 1.0]

# Orientation options
FLIP_NORMALS = False   # reverse u direction
SWAP_UV      = False   # swap u <-> v
# ─────────────────────────────────────────────────────────────────────────────

state = load_surface(
    SURFACE_NAME, EXPERIMENT_NAME,
    u_sample_num=U_SAMPLE, v_sample_num=V_SAMPLE,
    interval_u=INTERVAL_U, interval_v=INTERVAL_V,
)

# Apply flip/swap if requested
if FLIP_NORMALS:
    cp = state.bspline.controlpoints
    state.bspline.controlpoints = cp[::-1, :, :]
    state.normalsFlipped = True
if SWAP_UV:
    state.bspline = state.bspline.clone()
    state.bspline.controlpoints = state.bspline.controlpoints.swapaxes(0, 1)
    state.uvSwapped = True

print(f"Surface: {SURFACE_NAME}")
print(f"Sampling: {U_SAMPLE}×{V_SAMPLE} on u∈{INTERVAL_U}, v∈{INTERVAL_V}")

Surface: CMC
Sampling: 25×25 on u∈[0.0, 1.0], v∈[0.0, 1.0]


In [3]:
import ipywidgets as widgets
from IPython.display import display

def _update_surface(u_min, u_max, v_min, v_max, u_n, v_n):
    if u_min >= u_max or v_min >= v_max:
        return
    sample_selection(state,
                     interval_u=[u_min, u_max],
                     interval_v=[v_min, v_max],
                     u_sample_num=u_n,
                     v_sample_num=v_n)
    V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
    K, H, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
    kv.register_surface_mesh("surface", V, F)
    kv.add_scalar_quantity("surface", "MeanCurvature",     H.ravel(), defined_on="vertices")
    kv.add_scalar_quantity("surface", "GaussianCurvature", K.ravel(), defined_on="vertices")

_sl = dict(continuous_update=False, style={"description_width": "80px"}, layout=widgets.Layout(width="340px"))
u_min_w = widgets.FloatSlider(value=0.0, min=0.0,  max=0.95, step=0.05, description="u min", **_sl)
u_max_w = widgets.FloatSlider(value=1.0, min=0.05, max=1.0,  step=0.05, description="u max", **_sl)
v_min_w = widgets.FloatSlider(value=0.0, min=0.0,  max=0.95, step=0.05, description="v min", **_sl)
v_max_w = widgets.FloatSlider(value=1.0, min=0.05, max=1.0,  step=0.05, description="v max", **_sl)
u_n_w   = widgets.IntSlider(value=U_SAMPLE, min=5, max=60, step=5, description="u samples", **_sl)
v_n_w   = widgets.IntSlider(value=V_SAMPLE, min=5, max=60, step=5, description="v samples", **_sl)

out = widgets.interactive_output(_update_surface, {
    "u_min": u_min_w, "u_max": u_max_w,
    "v_min": v_min_w, "v_max": v_max_w,
    "u_n":   u_n_w,   "v_n":   v_n_w,
})
ui = widgets.VBox([
    widgets.HBox([u_min_w, u_max_w]),
    widgets.HBox([v_min_w, v_max_w]),
    widgets.HBox([u_n_w,   v_n_w]),
])
display(ui, out)
kv.show()

Output()

[kayviz] Opening http://127.0.0.1:8001/?app=script


---
## 1b — B-spline optimisation (enforce H ≠ 0)

Slightly deforms the surface control points to ensure the mean curvature H has
a consistent nonzero sign throughout the domain. This makes the sphere-congruence
initialisation r = 1/H well-conditioned.

Skip this cell if the surface already has |H| > 0 everywhere.

In [4]:
from moebius.pipeline import setup_bspline_optimizer, run_bspline_optimizer

# ── parameters ────────────────────────────────────────────────────────────────
H_THRESH     = 0.05
W_H          = 1.0
W_STEP       = 0.1
BSPLINE_ITER = 30
# ─────────────────────────────────────────────────────────────────────────────

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H_before, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
print(f"H before  min={H_before.min():.4f}  max={H_before.max():.4f}  "
      f"zero-crossings≈{int((np.diff(np.sign(H_before.ravel())) != 0).sum())}")

setup_bspline_optimizer(state, H_thresh=H_THRESH, w_H=W_H, w_step=W_STEP)
run_bspline_optimizer(state, max_iter=BSPLINE_ITER)

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H_after, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
print(f"H after   min={H_after.min():.4f}  max={H_after.max():.4f}  "
      f"zero-crossings≈{int((np.diff(np.sign(H_after.ravel())) != 0).sum())}")

kv.register_surface_mesh("surface", V, F)
kv.add_scalar_quantity("surface", "H_before", H_before.ravel(), defined_on="vertices")
kv.add_scalar_quantity("surface", "H_after",  H_after.ravel(),  defined_on="vertices")
kv.show()

H before  min=0.9257  max=1.2809  zero-crossings≈0
Thread pool: 3 workers (terms=3, cpus=10)
Starting fixed-damping LM optimization (max_iter=30, tol=1e-08)
Levenberg-Marquardt Optimization (Simple)
Iter   | Energy        | Norm Energy   | ||dx||        | μ             | ρ        | Time    
------------------------------------------------------------------------------------------
1      | 1.992841e-02  | 0.000000e+00  | 1.136735e-01  | 1.000000e-06  | N/A      | 0.0034  
10     | 3.899258e-03  | 0.000000e+00  | 2.313785e-02  | 1.000000e-06  | N/A      | 0.0031  
Converged at iteration 12 with energy 0.000000e+00

 WEIGHTS REPORT
Term                        Weight
----------------------------------
MeanCurvatureBspline    1.0000e+00
cp_surf_step_control    1.0000e-01
Lap_Fairness_cp_surf    0.0000e+00
 Total Iterations: 12
 Time per Iteration: 0.0029 seconds
 ENERGY REPORT
Term                          Energy      Normalized  Contrib.
----------------------------------------------------

---
## 2 — Sphere-congruence initialisation

In [5]:
state.init_mode = 0

compute_init(state)

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
r_vals = bisplev(state.u_pts, state.v_pts, state.r_uv)
n_flat = state.n_surf.reshape(-1, 3)
Vc = V + r_vals.ravel()[:, None] * n_flat

kv.register_surface_mesh("sphere_centers_init", Vc, F)
kv.register_surface_mesh("surface", V, F)
kv.add_vector_quantity("surface", "l_init", state.l_init.reshape(-1, 3), defined_on="vertices")
kv.show()

Init  r range  [0.7908, 1.0542]
Init  l·n mean  0.8614
[kayviz] Viewer already open — http://127.0.0.1:8001/?app=script


---
## 3a — Line-congruence optimisation

In [6]:
# ── parameters ────────────────────────────────────────────────────────────────
LC_MAX_ITER  = 60
ANGLE_LC_DEG = 85.0   # max angle l↔n  (degrees; 90 = l must be tangent)
W_LC         = 1.0
W_ORTH       = 1.0
# ─────────────────────────────────────────────────────────────────────────────

state.angle_lc = ANGLE_LC_DEG

setup_lc_optimizer(state, w_lc=W_LC, w_orth=W_ORTH)
run_lc_optimizer(state, max_iter=LC_MAX_ITER)

Mesh |V|=625, |F|=576, |E|=1200
Adding quadratic fairness for 529 vertices with 4 neighbors
Adding Laplacian fairness for 96 
Thread pool: 5 workers (terms=5, cpus=10)
Starting fixed-damping LM optimization (max_iter=60, tol=1e-08)
Levenberg-Marquardt Optimization (Simple)
Iter   | Energy        | Norm Energy   | ||dx||        | μ             | ρ        | Time    
------------------------------------------------------------------------------------------
1      | 3.903918e+09  | 6.246268e+06  | 6.247936e+02  | 1.000000e-02  | N/A      | 0.3624  
10     | 1.167319e-03  | 1.010351e-09  | 1.288460e-02  | 1.000000e-02  | N/A      | 0.3709  
Converged at iteration 18 with energy 8.286180e-08

 WEIGHTS REPORT
Term                  Weight
----------------------------
LineCong          1.0000e+00
LineCongOrth      1.0000e+00
l_unit            1.0000e+01
QuadFairness_l    0.0000e+00
Lap_Fairness_l    0.0000e+00
 Total Iterations: 18
 Time per Iteration: 0.2311 seconds
 ENERGY REPORT
Term        

In [7]:
l_opt, cp_opt = state.lc_opt.unpack("l", "rij")
state.r_uv[2] = cp_opt
r_vals = bisplev(state.u_pts, state.v_pts, state.r_uv)
Vc = V + r_vals.ravel()[:, None] * n_flat

l_opt   = l_opt.reshape(-1, 3)
angle_n = np.degrees(np.arccos(np.clip(np.einsum("ij,ij->i", l_opt, n_flat), -1, 1)))

kv.register_surface_mesh("surface",        V, F)
kv.register_surface_mesh("sphere_centers", Vc, F)
kv.add_vector_quantity("surface", "l_opt",    l_opt,   defined_on="vertices")
kv.add_scalar_quantity("surface", "angle_l_n", angle_n, defined_on="vertices")
kv.show()

[kayviz] Viewer already open — http://127.0.0.1:8001/?app=script


---
## 3b — Torsal optimisation

In [8]:
# ── parameters ────────────────────────────────────────────────────────────────
TORSAL_MAX_ITER  = 100
ANGLE_TORSAL_DEG = 30.0   # minimum angle between torsal planes
W_TORSAL         = 1.0
W_TANGLE         = 0.5
# ─────────────────────────────────────────────────────────────────────────────

state.angle_torsal = ANGLE_TORSAL_DEG

setup_torsal_optimizer(state, w_lc=W_LC, w_orth=W_ORTH,
                        w_torsal=W_TORSAL, w_tangle=W_TANGLE)
run_torsal_optimizer(state, max_iter=TORSAL_MAX_ITER)

Mesh |V|=625, |F|=576, |E|=1200
Adding quadratic fairness for 529 vertices with 4 neighbors
Adding Laplacian fairness for 96 
Thread pool: 10 workers (terms=11, cpus=10)
Starting fixed-damping LM optimization (max_iter=100, tol=1e-08)
Levenberg-Marquardt Optimization (Simple)
Iter   | Energy        | Norm Energy   | ||dx||        | μ             | ρ        | Time    
------------------------------------------------------------------------------------------
1      | 1.529196e+02  | 2.654622e-01  | 1.716734e+01  | 1.054858e-02  | N/A      | 1.8642  
10     | 1.725560e+00  | 1.295087e-03  | 4.172214e+00  | 1.054858e-02  | N/A      | 1.7976  
20     | 8.170312e-02  | 7.104888e-05  | 6.352871e-01  | 1.054858e-02  | N/A      | 1.7287  
30     | 4.140707e-02  | 3.055720e-05  | 1.950733e-01  | 1.054858e-02  | N/A      | 1.5165  
40     | 3.293650e-02  | 2.360590e-05  | 1.078030e-01  | 1.054858e-02  | N/A      | 1.6001  
50     | 2.792169e-02  | 1.998638e-05  | 8.459473e-02  | 1.054858e-02  | N

In [9]:
from moebius.utils.mesh_utils import lc_info_at_grid_points, vec_dot

l_t, u1, v1, u2, v2, nt1, nt2 = state.torsal_opt.unpack(
    "l", "u1", "v1", "u2", "v2", "nt1", "nt2")

l_t  = l_t.reshape(state.u_sample_num, state.v_sample_num, 3)
lc, lu, lv = lc_info_at_grid_points(l_t)
lu = lu.reshape(-1, 3); lv = lv.reshape(-1, 3)
t1 = u1[:, None] * lu + v1[:, None] * lv
t2 = u2[:, None] * lu + v2[:, None] * lv

nt1 = nt1.reshape(-1, 3)
nt2 = nt2.reshape(-1, 3)
torsal_angle = np.degrees(np.arccos(np.clip(np.abs(vec_dot(nt1, nt2)), 0, 1)))

# Face barycenters and normalised directions
bary = (V[F[:, 0]] + V[F[:, 1]] + V[F[:, 2]] + V[F[:, 3]]) / 4
t1_n = t1 / (np.linalg.norm(t1, axis=1, keepdims=True) + 1e-12)
t2_n = t2 / (np.linalg.norm(t2, axis=1, keepdims=True) + 1e-12)

# Scale arrows to ~1/4 of mean face diagonal
diag = np.mean(np.linalg.norm(V[F[:, 2]] - V[F[:, 0]], axis=1))
s = diag / 4

# Build curve-network for each torsal direction:
# each face contributes one centred segment  [bary - s*t, bary + s*t]
def _torsal_curves(bary, t, scale):
    F_cnt = len(bary)
    verts = np.empty((2 * F_cnt, 3))
    verts[0::2] = bary - scale * t
    verts[1::2] = bary + scale * t
    edges = np.stack([np.arange(0, 2*F_cnt, 2),
                      np.arange(1, 2*F_cnt, 2)], axis=1)
    return verts, edges

v1_net, e1_net = _torsal_curves(bary, t1_n, s)
v2_net, e2_net = _torsal_curves(bary, t2_n, s)

kv.register_surface_mesh("surface", V, F)
kv.add_scalar_quantity("surface", "torsal_angle", torsal_angle, defined_on="faces")
kv.add_vector_quantity("surface", "l", l_t.reshape(-1, 3), defined_on="vertices")

kv.register_curve_network("torsal_t1", v1_net, e1_net, color=(0.9, 0.2, 0.1), radius=0.002)
kv.register_curve_network("torsal_t2", v2_net, e2_net, color=(0.1, 0.3, 0.9), radius=0.002)
kv.show()

[kayviz] Viewer already open — http://127.0.0.1:8001/?app=script


---
## 4 — Remeshing

In [10]:
# ── export cross-field for remesher ──────────────────────────────────────────
save_remesh_input(state)
print("Cross-field saved. Now run the remesher externally, or use run_remesher() below.")

Remesh input saved to /Users/cisneras/KAUST/Moebius_project/notebooks/out/CMC/experiment0/remesh_input.obj
Cross-field saved. Now run the remesher externally, or use run_remesher() below.


In [11]:
# ── option A: call the remesher from here ────────────────────────────────────
N_FACES_TARGET = 400
# state.remesher_bin = "/path/to/quadRemesher"   # override if needed

ret = run_remesher(state, n_faces=N_FACES_TARGET)
if ret == 0:
    load_remeshed_surface(state)
    kv.register_surface_mesh("remeshed", state.V_remesh, state.F_remesh)
    kv.show()
else:
    print(f"Remesher returned exit code {ret}. ")
    print("You can load the remeshed OBJ manually:")
    print(f"  load_remeshed_surface(state)")

Running: /Users/cisneras/KAUST/QuadRemesher/Quad_Remesher/build/quadRemesher /Users/cisneras/KAUST/Moebius_project/notebooks/out/CMC/experiment0/remesh_input.obj /Users/cisneras/KAUST/Moebius_project/notebooks/out/CMC/experiment0/crossfield.txt /Users/cisneras/KAUST/Moebius_project/notebooks/out/CMC/experiment0/remeshed.obj 400
Loaded mesh: 625 vertices, 576 faces
Remesher returned exit code 1. 
You can load the remeshed OBJ manually:
  load_remeshed_surface(state)


IOError: readDMAT() first row should be [num cols] [num rows]...
Error: Could not read direction files.


In [ ]:
# ── option B: load previously remeshed OBJ ───────────────────────────────────
# load_remeshed_surface(state)
# kv.register_surface_mesh("remeshed", state.V_remesh, state.F_remesh)
# kv.show()

---
## 5 — Post-optimisation

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
POSTOPT_MAX_ITER = 60
W_SPHERE   = 1.0
W_SUPPORT  = 1e-1
W_REG      = 1e-2
# ─────────────────────────────────────────────────────────────────────────────

setup_postopt_optimizer(state,
                         w_sphere=W_SPHERE,
                         w_support=W_SUPPORT,
                         w_reg=W_REG)
run_postopt_optimizer(state, max_iter=POSTOPT_MAX_ITER)

In [ ]:
# ── visualise post-opt result ─────────────────────────────────────────────────
V_opt = state.postopt.unpack("v").reshape(-1, 3)
A, B, C = state.postopt.unpack("A", "B", "C")
B_r = B.reshape(-1, 3)
centers, radii = sphere_centers_radii_from_params(A, B_r, C)

# Spherical panels
pV, pF = spherical_panels_from_mesh(V_opt, state.F_remesh, centers, radii, n_arc=12)

kv.register_surface_mesh("optimised_mesh",  V_opt,  state.F_remesh)
kv.register_surface_mesh("sphere_panels",   pV,     pF)
kv.add_scalar_quantity("optimised_mesh", "sphere_radius", radii, defined_on="faces")
kv.show()

---
## 6 — Save results

In [ ]:
save_results(state)
print("Saved to:", state.save_path)

---
## Utilities — reload & inspect

```python
# Reload a saved state
state = load_state("../notebooks/out/CMC/experiment0/state.pickle")

# Plot energy history
plt.figure(figsize=(8, 4))
plt.semilogy(state.lc_opt.energy,     label="LC")
plt.semilogy(state.torsal_opt.energy, label="Torsal")
plt.legend(); plt.xlabel("Iteration"); plt.ylabel("Energy"); plt.grid()
```